<a href="https://colab.research.google.com/github/its-bhavya/tos_classifier/blob/main/notebooks/01_eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# CLASS DISTRIBUTION AND IMBALANCE RATIO

In [ ]:
# 1. Load the clean dataset
df = pd.read_csv('clausess.csv')

# 2. Calculate Counts and Percentages
counts = df['label'].value_counts()
percentages = df['label'].value_counts(normalize=True) * 100

print("--- Class Counts ---")
print(counts)
print("\n--- Class Percentages ---")
print(percentages.round(2).astype(str) + '%')

# 3. Calculate Imbalance Ratio (Majority vs Minority)
majority_class = counts.idxmax()
minority_class = counts.idxmin()
imbalance_ratio = counts.max() / counts.min()

print(f"\nImbalance Ratio ({majority_class} vs {minority_class}): {imbalance_ratio:.2f}:1")

# 4. Generate Plot
plt.figure(figsize=(8, 5))
ax = sns.barplot(x=counts.index, y=counts.values, palette='mako')

plt.title('Class Distribution of ToS Clauses', fontsize=14, pad=15, fontweight='bold')
plt.xlabel('Risk Category (Label)', fontsize=12)
plt.ylabel('Number of Clauses', fontsize=12)

# Add exact numbers on top of the bars for clarity
for p in ax.patches:
    ax.annotate(f"{int(p.get_height())}",
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center',
                xytext=(0, 9), textcoords='offset points')

# Save the plot
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=300)
plt.show()

# CLAUSE LENGTH DISTRIBUTION

In [ ]:
# 1. Load data and standardize column names
if 'title' in df.columns:
    df.rename(columns={'title': 'clause_text'}, inplace=True)
df = df.dropna(subset=['clause_text'])

# 2. Calculate word counts
df['word_count'] = df['clause_text'].apply(lambda x: len(str(x).split()))

# 3. Plot the distribution
plt.figure(figsize=(10, 6))
sns.histplot(data=df, x='word_count', hue='label',
             palette={'good': '#2ecc71', 'neutral': '#95a5a6', 'bad': '#e74c3c'},
             element='step', stat='density', common_norm=False, kde=True)

plt.title('Clause Length Distribution by Risk Category', fontsize=14, fontweight='bold')
plt.xlabel('Number of Words in Clause', fontsize=12)
plt.ylabel('Density', fontsize=12)
plt.xlim(0, 100) # Capping at 100 words for better visualization of the bulk data
plt.tight_layout()

# Save and show
plt.savefig('clause_length_distribution.png', dpi=300)
plt.show()

# Print the averages to note down
print(df.groupby('label')['word_count'].mean().round(2))

# GENERATE WORD COUNTS

In [ ]:
from wordcloud import WordCloud

# Set up a 1x3 grid for our three classes
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
categories = ['good', 'neutral', 'bad']
colors = ['Greens', 'Greys', 'Reds']

for i, (cat, cmap) in enumerate(zip(categories, colors)):
    # Combine all text for this category
    text = " ".join(clause for clause in df[df['label'] == cat]['clause_text'])

    # Generate the word cloud
    wordcloud = WordCloud(width=800, height=800,
                          background_color='white',
                          colormap=cmap,
                          max_words=100).generate(text)

    # Plot in the correct subplot
    axes[i].imshow(wordcloud, interpolation='bilinear')
    axes[i].set_title(f"'{cat.upper()}' Clauses", fontsize=16, fontweight='bold', pad=15)
    axes[i].axis('off')

plt.tight_layout()
plt.savefig('wordclouds_by_class.png', dpi=300)
plt.show()

# EXTRACT TOP 20 UNIGRAMS AND BIGRAMS

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

def get_top_n_grams(corpus, n=20, n_gram_range=(1,1)):
    vec = CountVectorizer(stop_words='english', ngram_range=n_gram_range).fit(corpus)
    bag_of_words = vec.transform(corpus)
    sum_words = bag_of_words.sum(axis=0)
    words_freq = [(word, sum_words[0, idx]) for word, idx in vec.vocabulary_.items()]
    words_freq = sorted(words_freq, key=lambda x: x[1], reverse=True)
    return words_freq[:n]

# Iterate through classes and print the top terms
for label in ['good', 'neutral', 'bad']:
    print(f"\n{'='*40}")
    print(f" CATEGORY: {label.upper()}")
    print(f"{'='*40}")

    corpus = df[df['label'] == label]['clause_text']

    unigrams = get_top_n_grams(corpus, n=20, n_gram_range=(1,1))
    bigrams = get_top_n_grams(corpus, n=20, n_gram_range=(2,2))

    print("\nTop 20 Unigrams:")
    for word, freq in unigrams:
        print(f" - {word}: {freq}")

    print("\nTop 20 Bigrams:")
    for word, freq in bigrams:
        print(f" - {word}: {freq}")

# NOTES

**The "Bad" Pattern:** There are bigrams like 'party cookies' and 'prior notice' popping up heavily in the bad clauses. This heavily points towards data-sharing and unilateral change risks.

**The Overlap:** Both 'Good' and 'Bad' classes heavily feature the bigram 'personal data'. This means the model won't be able to rely on just those two words; it will have to learn the context around them (e.g., "protects personal data" vs "sells personal data"). This is a great justification for using TF-IDF with unigrams+bigrams (context over single tokens) rather than a simple keyword detector.